# The regime feature space (Steps 2-8)

A compact set of causal features, one per phenomenon, each with a documented reason; near-duplicates (|Spearman| >= 0.98) may not both enter a model. Scalers are fitted on training rows only; missing features are marginalised, never zero-filled.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.regimes` and `xauusd_quant.research.regime_*` and read back from `results/regime_research/`. Generate those first:

```
xq regime-research --timeframe 1h --model hmm --states 3
xq regime-research --timeframe 15m --model gmm --components 4
xq regime-walk-forward --timeframe 5m --model hmm --states 4
xq regime-research --all --resume        # the full grid (hours; see --dry-run)
xq regime-research --controls            # synthetic controls
xq regime-research --compare             # cross-timeframe tables
```

**Causal vs offline.** Walk-forward results use models fitted on earlier data only, a frozen scaler and the HMM *forward filter* P(S_t | X_<=t): they are causal and the only thing stored under `data/features/regime/`. Full-sample fits, smoothed probabilities and Viterbi paths are **OFFLINE / NON-CAUSAL RESEARCH ONLY** and never features. States are numbered (`state_0`, `state_1`, ...), never named; their descriptions come from the profiles. Nothing here is a signal.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.regimes import load_regime_config

regime = load_regime_config(ROOT / "config" / "regimes.yaml")

TIMEFRAME = "1h"       # 5m | 15m | 30m | 1h - change and re-run
MODEL = "hmm"          # kmeans | gmm | gmm_diag | hmm
K = 3                  # number of states

study = regime.results_dir(TIMEFRAME, MODEL, K)
tf_dir = regime.timeframe_dir(TIMEFRAME)
if not study.exists():
    print(f"No results for {TIMEFRAME} {MODEL} K={K}. Run:  xq regime-research "
          f"--timeframe {TIMEFRAME} --model {MODEL} --states {K}")

def table(name, folder=None):
    """Load a previously generated regime table (None if it was not written)."""
    path = (folder or study) / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name, folder=None):
    path = (folder or study / "plots") / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

summary_path = study / "summary.json"
summary = json.loads(summary_path.read_text())["summary"] if summary_path.exists() else {}
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Feature manifest: why each feature is in (or out)

In [ ]:
manifest = json.loads((tf_dir / 'feature_manifest.json').read_text())
pl.DataFrame([{'feature': k, **{kk: vv for kk, vv in v.items() if kk != 'rationale'}}
              for k, v in manifest['features'].items()])

In [ ]:
for name, reason in manifest['excluded'].items():
    print(f'{name}: {reason}')

## Redundancy between the inputs

In [ ]:
pl.read_csv(tf_dir / 'feature_redundancy.csv').sort(pl.col('spearman').abs(), descending=True).head(15)

## Coverage and the integrity gate

In [ ]:
print('complete rows:', manifest['complete_rows_share'])
print(json.dumps(manifest['integrity_gate'], indent=1)[:3000])

## Robust vs standard scaling (Step 7)

In [ ]:
rows = []
for path in sorted((tf_dir / 'offline').glob('scaling_comparison_*.json')):
    rows += json.loads(path.read_text())['rows']
pl.DataFrame(rows).drop('clipped_share_by_feature') if rows else None